In [7]:
import requests
import numpy as np
import pandas as pd
import sqlite3
from datetime import datetime, timezone

In [8]:
conn = sqlite3.connect("screener.db", timeout=20)
cursor = conn.cursor()
conn.execute("PRAGMA journal_mode=WAL;")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS pool_snapshots (
        id               INTEGER PRIMARY KEY AUTOINCREMENT,
        pair_address     TEXT REFERENCES pools(pair_address),
        price_usd        REAL,
        liquidity_usd    REAL,
        volume_m5        REAL,
        volume_h1        REAL,
        volume_h24       REAL,
        price_change_m5  REAL,
        price_change_h1  REAL,
        price_change_h24 REAL,
        market_cap       REAL,
        fdv              REAL,
        snapshot_at      TIMESTAMP
    )
""")
conn.commit()

In [9]:
cursor.execute("""
    SELECT pair_address, token_address, chain_id
    FROM pools
""")
pools = cursor.fetchall()
snapshot_count = 0

for pair_address, token_address, chain_id in pools:

    response = requests.get(
        f"https://api.dexscreener.com/token-pairs/v1/{chain_id}/{token_address}"
    )

    pair_response = response.json()
    pool = next((p for p in pair_response if p.get("pairAddress") == pair_address), None)

    if pool is None:
        continue

    now = datetime.now(timezone.utc).isoformat()

    cursor.execute(
        """
        INSERT INTO pool_snapshots(
            pair_address,
            price_usd,
            liquidity_usd,
            volume_m5,
            volume_h1,
            volume_h24,
            price_change_m5,
            price_change_h1,
            price_change_h24,
            market_cap,
            fdv,
            snapshot_at
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            pair_address,
            pool.get("priceUsd"),
            pool.get("liquidity", {}).get("usd"),
            pool.get("volume", {}).get("m5"),
            pool.get("volume", {}).get("h1"),
            pool.get("volume", {}).get("h24"),
            pool.get("priceChange", {}).get("m5"),
            pool.get("priceChange", {}).get("h1"),
            pool.get("priceChange", {}).get("h24"),
            pool.get("marketCap"),
            pool.get("fdv"),
            now,
        ),
    )
    snapshot_count += 1

conn.commit()
print(f"Snapshots inserted: {snapshot_count}")
conn.close()

Snapshots inserted: 93


In [10]:
%load_ext sql
%sql sqlite:///screener.db

Connecting to 'sqlite:///screener.db'

In [11]:
%%sql
SELECT * FROM pool_snapshots LIMIT 5

Running query in 'sqlite:///screener.db'

id,pair_address,price_usd,liquidity_usd,volume_m5,volume_h1,volume_h24,price_change_m5,price_change_h1,price_change_h24,market_cap,fdv,snapshot_at
1,0xE520a7C2d2Ed54FA9d50Cf2BAf3969148BbdF46b,0.1425,86.63,0.55,0.55,1.12,2.35,2.35,4.8,44.0,44.0,2026-06-28T12:34:45.263602+00:00
2,C1KHGdzEh3DVr7iE8tUiqUuox9eYigLFBhikpU3rbjwD,5.699e-05,17092.78,35956.79,46431.74,46431.74,10.88,85.94,85.94,56995.0,56995.0,2026-06-28T12:34:45.688902+00:00
3,3ipHnTpz72RGsQgGPJ54PNqdN2Svne98akGZAgUye3hw,0.0001068,117.22,63.47,63.47,63.47,10.46,10.46,10.46,106894.0,106894.0,2026-06-28T12:34:45.853068+00:00
4,BajPdk4gBPtBohUQNofoYKLfpdEi9yRRHpioEzBeDXBZ,9.41e-05,30.12,27.68,27.68,27.68,-12.54,-12.54,-12.54,94110.0,94110.0,2026-06-28T12:34:45.997022+00:00
5,D216BgduYfZodoyPPLcN2tqY78YcD2GfhbyDXVufj9XY,0.0001102,4.03,1.19,1.19,1.19,111.0,111.0,111.0,110238.0,110238.0,2026-06-28T12:34:46.147843+00:00
